In [36]:
import os
import json
import time
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# Setup
client = OpenAI()
MODEL_NAME = "gpt-4o"
TEMPERATURE = 0.2

In [37]:
SCHEMA = {
    "type": "object",
    "properties": {
        "metric_human_name": {
            "type": "string",
            "description": "A simple, human-readable title for the section (e.g. 'Direct Impact')"
        },
        "question_1_qualitative": {
            "type": "object",
            "properties": {
                "question_text": {"type": "string"},
                "helper_text": {"type": "string", "description": "The 'For example...' text"}
            },
            "required": ["question_text", "helper_text"],
            "additionalProperties": False
        },
        "question_2_quantitative": {
            "type": "object",
            "properties": {
                "question_text": {"type": "string"},
                "constraint_note": {"type": "string", "description": "What to EXCLUDE based on rules"},
                "unit": {"type": "string"}
            },
            "required": ["question_text", "constraint_note", "unit"],
            "additionalProperties": False
        },
        "question_3_verification": {
            "type": "object",
            "properties": {
                "question_text": {"type": "string"},
                "guidance_text": {"type": "string", "description": "Examples of valid verification proofs"}
            },
            "required": ["question_text", "guidance_text"],
            "additionalProperties": False
        }
    },
    "required": ["metric_human_name", "question_1_qualitative", "question_2_quantitative", "question_3_verification"],
    "additionalProperties": False
}

In [38]:
kpi_df = pd.read_csv("./data/Hult Prize - Form Questions - Sheet2.csv")

In [39]:
with open("./prompt_templates/hult_form_questions.md", "r") as f:
    PROMPT_TEMPLATE = f.read()

In [40]:
def generate_form_questions(kpi_id: str, kpi_definition: str, calculation_method: str) -> dict:
    
    prompt = PROMPT_TEMPLATE.replace("{{KPI_ID}}", kpi_id)
    prompt = prompt.replace("{{KPI_Definition}}", kpi_definition)
    prompt = prompt.replace("{{Calculation_Method_Examples}}", calculation_method)
    
    # Call API with structured output
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=TEMPERATURE,
        messages=[{"role": "user", "content": prompt}],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "form_questions",
                "strict": True,
                "schema": SCHEMA
            }
        }
    )
    
    # Parse and return
    return json.loads(response.choices[0].message.content)

In [41]:
results = []

for idx, row in kpi_df.iterrows():
    kpi_id = row["KPI ID"]
    print(f"Processing {kpi_id}...")
    
    result = generate_form_questions(
        kpi_id=kpi_id,
        kpi_definition=row["KPI Definition"],
        calculation_method=row["Calculation Method & Examples"]
    )
    result["kpi_id"] = kpi_id
    result["metric_group"] = row["Metric Group"]
    results.append(result)

print(f"\n Generated questions for {len(results)} KPIs")

Processing P-IM-1...
Processing P-IM-4...
Processing P-IM-2...
Processing P-IM-3...
Processing P-IM-5...
Processing P-IM-6...
Processing P-IM-7...
Processing P-IM-8...
Processing P-IM-9...
Processing P-I-1...
Processing P-I-2...
Processing P-1-3...
Processing P-T-3...
Processing P-T-6...
Processing P-T-4...
Processing P-T-5...
Processing P-T-2...
Processing P-T-1...
Processing P-T-7...
Processing P-T-8...

 Generated questions for 20 KPIs


In [42]:
output_path = "./output/hult/hult_form_questions_4o.json"

with open(output_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved to {output_path}")

Saved to ./output/hult/hult_form_questions_4o.json


In [43]:
def result_to_markdown(r: dict) -> str:
    """Convert a single result to markdown format."""
    md = f"## {r['metric_human_name']} ({r['kpi_id']})\n\n"
    md += f"**Metric Group:** {r['metric_group']}\n\n"
    
    q1 = r["question_1_qualitative"]
    md += f"### Question 1: Definition & Narrative\n"
    md += f"**{q1['question_text']}**\n\n"
    md += f"*Helper:* {q1['helper_text']}\n\n"
    
    q2 = r["question_2_quantitative"]
    md += f"### Question 2: The Data Point\n"
    md += f"**{q2['question_text']}**\n\n"
    md += f"*Constraint:* {q2['constraint_note']}\n\n"
    md += f"*Unit:* {q2['unit']}\n\n"
    
    q3 = r["question_3_verification"]
    md += f"### Question 3: Verification\n"
    md += f"**{q3['question_text']}**\n\n"
    md += f"*Guidance:* {q3['guidance_text']}\n\n"
    
    md += "---\n\n"
    return md

# Generate full markdown
markdown_content = "# Hult Prize - Generated Form Questions\n\n"
for r in results:
    markdown_content += result_to_markdown(r)

# Save
md_path = "./output/hult/hult_form_questions_4o.md"
with open(md_path, "w") as f:
    f.write(markdown_content)

print(f"Saved to {md_path}")

Saved to ./output/hult/hult_form_questions_4o.md


In [44]:
# Preview first result
print(json.dumps(results[0], indent=2))

{
  "metric_human_name": "Direct Impact",
  "question_1_qualitative": {
    "question_text": "What does a successful outcome look like for someone using your solution? Describe a specific change or improvement they experience.",
    "helper_text": "For example, looking at the definition provided: In Education, if 500 students were served and 320 improved their scores by 15% or more, the successful outcome is the score improvement. What is the specific outcome that counts as success for you?"
  },
  "question_2_quantitative": {
    "question_text": "How many people have achieved this successful outcome since you started?",
    "constraint_note": "Do not count people who only signed up or were reached but did not achieve the intended outcome.",
    "unit": "people"
  },
  "question_3_verification": {
    "question_text": "How do you track or verify the number of people who achieved this outcome?",
    "guidance_text": "Examples: You might use attendance sheets, CRM logs, or follow-up sur

In [45]:
def result_to_cell_text(r: dict) -> str:
    """Format all questions for a single KPI into one cell."""
    lines = []
    
    # Question 1
    q1 = r["question_1_qualitative"]
    lines.append("Question No. 1: Definition & Narrative")
    lines.append(q1["question_text"])
    lines.append(f"_Helper: {q1['helper_text']}_")
    lines.append("Input Type: Text (Open-ended)")
    lines.append("")
    
    # Question 2
    q2 = r["question_2_quantitative"]
    lines.append("Question No. 2: The Data Point")
    lines.append(q2["question_text"])
    lines.append(f"_Constraint: {q2['constraint_note']}_")
    lines.append(f"Input Type: Number ({q2['unit']})")
    lines.append("")
    
    # Question 3
    q3 = r["question_3_verification"]
    lines.append("Question No. 3: Verification")
    lines.append(q3["question_text"])
    lines.append(f"_Guidance: {q3['guidance_text']}_")
    lines.append("Input Type: Text (Open-ended)")
    
    return "\n".join(lines)

# Build DataFrame
csv_data = []
for r in results:
    csv_data.append({
        "KPI ID": r["kpi_id"],
        "Metric Group": r["metric_group"],
        "Metric Name": r["metric_human_name"],
        "Form Questions": result_to_cell_text(r)
    })

df_output = pd.DataFrame(csv_data)

# Save to CSV
csv_path = "./output/hult/hult_form_questions.csv"
df_output.to_csv(csv_path, index=False)

print(f"Saved to {csv_path}")
df_output.head()

Saved to ./output/hult/hult_form_questions.csv


,KPI ID,Metric Group,Metric Name,Form Questions
0,P-IM-1,Impact (IM),Direct Impact,Question No. 1: Definition & Narrative\nWhat d...
1,P-IM-4,Impact (IM),Indirect Impact,Question No. 1: Definition & Narrative\nHow do...
2,P-IM-2,Impact (IM),Direct Job Creation,Question No. 1: Definition & Narrative\nHow do...
3,P-IM-3,Impact (IM),Indirect Job Creation,Question No. 1: Definition & Narrative\nHow do...
4,P-IM-5,Impact (IM),Carbon Emissions Impact,Question No. 1: Definition & Narrative\nHow do...
